<hr style="border:2px solid #0281c9"> </hr>

<img align="left" alt="ESO Logo" src="http://archive.eso.org/i/esologo.png">  

<div align="center">
  <h1 style="color: #0281c9; font-weight: bold;">ESO Science Archive</h1> 
  <h2 style="color: #0281c9; font-weight: bold;">Jupyter Notebooks</h2>
</div>

<hr style="border:2px solid #0281c9"> </hr>

# **Catalogue Data: Query with TAP**

This notebook demonstrates how to query ESO catalogue tables using free-form ADQL via `astroquery.eso`. The examples reflect common TAP workflows, including:

- schema discovery (`TAP_SCHEMA`)
- spatial filtering (e.g. cone searches)
- metadata-driven joins
- light-curve retrieval

Additional examples are available in the ESO documentation:

- https://archive.eso.org/tap_cat/examples  
- https://archive.eso.org/programmatic/#TAP

By default, `eso.query_tap()` connects to the observations TAP service (`tap_obs`). 

For catalogue queries, you must explicitly set:
```python
eso.query_tap("...", tap_endpoint="tap_cat") 
```

# **Importing and basic usage of astroquery.eso**

In [1]:
import astroquery # import astroquery
print(f"astroquery version: {astroquery.__version__}") # check the version of astroquery

astroquery version: 0.4.12.dev505+gf2a77a615.d20260427


In [2]:
from astroquery.eso import Eso 
from astropy.coordinates import SkyCoord 
import astropy.units as u

In [3]:
eso = Eso() # create an instance of the ESO class

# **Basic TAP usage**

A simple metadata query against `TAP_SCHEMA.tables` to check which are avalible.

Also a good way to confirm that we are quering to the catalogue TAP service (i.e. with `tap_endpoint="tap_cat"`) rather than the observations TAP service (i.e. with `tap_endpoint="tap_obs"`).

In [4]:
query = "SELECT table_name FROM TAP_SCHEMA.tables WHERE schema_name = 'safcat' ORDER BY table_name"
table = eso.query_tap(query, tap_endpoint="tap_cat")
table[:3]

table_name
object
AMBRE_HARPS_V1
AMBRE_UVES_V1
AMBRE_V1


# **Schema discovery examples**

The catalogue TAP metadata tables are useful for discovering what is available before writing a science query - should give same as `eso.list_catalogs(all_versions=True)`. 

In [5]:
query = """
        SELECT table_name, cat_id, rel_descr_url
        FROM TAP_SCHEMA.tables
        WHERE schema_name = 'safcat' AND cat_id IS NOT NULL
        ORDER BY cat_id
        """

release_docs = eso.query_tap(query, tap_endpoint="tap_cat")
release_docs[:3]

table_name,cat_id,rel_descr_url
object,int32,object
AMBRE_V1,13,http://www.eso.org/rm/api/v1/public/releaseDescriptions/7
GOODS_FORS2_V1,31,http://www.eso.org/rm/api/v1/public/releaseDescriptions/37
HUGS_GOODSS_K_V1,32,http://www.eso.org/rm/api/v1/public/releaseDescriptions/48


Query column metadata for a specific table, e.g. `COSMOS2015_Laigle_v1_1b_latestV7_fits_V1`. 

In [6]:
query = """
        SELECT column_name, datatype, unit, ucd
        FROM TAP_SCHEMA.columns
        WHERE table_name = 'COSMOS2015_Laigle_v1_1b_latestV7_fits_V1'
        ORDER BY column_name
        """

cosmos_columns = eso.query_tap(query, tap_endpoint="tap_cat")
cosmos_columns[:3]

column_name,datatype,unit,ucd
object,object,object,object
AGE,DOUBLE,,time.age
ALPHA_J2000,DOUBLE,deg,pos.eq.ra;meta.main
B_FLAGS,SMALLINT,mag,meta.code;phot.mag;em.opt.B


Find which columns of a specific table have a particular UCD, e.g. `pos.eq.ra;meta.main` and `pos.eq.dec;meta.main` for the main RA and Dec columns.

In [13]:
query = """
        SELECT column_name
        FROM TAP_SCHEMA.columns
        WHERE table_name = 'PESSTO_TRAN_CAT_fits_V2'
        AND ucd in ('pos.eq.ra;meta.main', 'pos.eq.dec;meta.main'
        )
        """

main_coords = eso.query_tap(query, tap_endpoint="tap_cat")
main_coords[:3]

column_name
object
TRANSIENT_RAJ2000
TRANSIENT_DECJ2000


# **Spatial query example**

Free ADQL also lets you perform cone searches directly. 

Here we resolve the galaxy `ESO 154-10`, convert the search radius to degrees, and then search the PESSTO transient catalogue with `CONTAINS(POINT, CIRCLE)`.

In [8]:
target = SkyCoord.from_name("ESO 154-10") # resolve the target coordinates online
ra_deg = target.icrs.ra.to_value(u.deg)
dec_deg = target.icrs.dec.to_value(u.deg)
radius_deg = (2.4 * u.arcmin).to_value(u.deg)

print(ra_deg, dec_deg, radius_deg)

41.2863167 -55.7405972 0.04


In [9]:
query = f"""
        SELECT host_id, transient_id, transient_classification,
            transient_raj2000, transient_decj2000
        FROM PESSTO_TRAN_CAT_fits_V2
        WHERE CONTAINS(
            POINT('ICRS', transient_raj2000, transient_decj2000),
            CIRCLE('ICRS', {ra_deg}, {dec_deg}, {radius_deg})
        ) = 1
        """

pessto_cone = eso.query_tap(query, tap_endpoint="tap_cat")
pessto_cone[:3]

HOST_ID,TRANSIENT_ID,TRANSIENT_CLASSIFICATION,TRANSIENT_RAJ2000,TRANSIENT_DECJ2000
,,,deg,deg
object,object,object,float32,float32
ESO 154- G 010,SN2014eg,SN Ia,41.288624,-55.73803
ESO 154- G 010,SN2013fc,SN IIn,41.287292,-55.740917


# **Metadata join example**

For some catalogue families, useful information is split across several related tables. The metadata join tables help reveal how those products are linked.

In [10]:
query = """
        SELECT k.from_table, kc.from_column, k.target_table, kc.target_column
        FROM TAP_SCHEMA.columns AS c, TAP_SCHEMA.keys AS k, TAP_SCHEMA.key_columns AS kc
        WHERE c.table_name = 'VVV_CAT_V2'
        AND (
            (k.from_table = c.table_name AND kc.from_column = c.column_name)
            OR
            (k.target_table = c.table_name AND kc.target_column = c.column_name)
        )
        AND k.key_id = kc.key_id
        ORDER BY table_name, column_name, 1, 3, 2
        """

vvv_join_keys = eso.query_tap(query, tap_endpoint="tap_cat")
vvv_join_keys[:3]

from_table,from_column,target_table,target_column
object,object,object,object
VVV_MPHOT_Ks_V2,SOURCEID,VVV_CAT_V2,SOURCEID
VVV_VAR_V2,SOURCEID,VVV_CAT_V2,SOURCEID


# **Light-curve query examples**

Once the join keys are known, you can write ADQL joins that connect a master catalogue to a multi-epoch or variability table.

In [11]:
query = """
        SELECT TOP 10 VVV.IAUNAME, VVV.SOURCEID, VVV.RA2000, VVV.DEC2000,
                    M.MJD, M.KSMAG, M.KSERR,
                    V.KSMEANMAG, V.KSAMPL, KSPROBVAR
        FROM VVV_CAT_V2 AS VVV
        JOIN VVV_MPHOT_Ks_V2 AS M ON VVV.SOURCEID = M.SOURCEID
        JOIN VVV_VAR_V2 AS V ON VVV.SOURCEID = V.SOURCEID
        WHERE VVV.IAUNAME = 'VVV J135433.73-594836.43'
        ORDER BY 1, 5
        """

vvv_lightcurve = eso.query_tap(query, tap_endpoint="tap_cat")
vvv_lightcurve[:3]

IAUNAME,SOURCEID,RA2000,DEC2000,MJD,KSMAG,KSERR,KSMEANMAG,KSAMPL,KSPROBVAR
,,Degrees,Degrees,day,mag,mag,mag,Mag,
object,int64,float64,float64,float64,float32,float32,float32,float32,float64
VVV J135433.73-594836.43,515827922228,208.64058136308523,-59.810121010900616,55260.30682830811,11.479307,0.001658409,11.391281,0.10365963,0.999999
VVV J135433.73-594836.43,515827922228,208.64058136308523,-59.810121010900616,55337.17406567468,11.392498,0.0016116726,11.391281,0.10365963,0.999999
VVV J135433.73-594836.43,515827922228,208.64058136308523,-59.810121010900616,55374.132435003914,11.391262,0.0016530175,11.391281,0.10365963,0.999999


In [12]:
radius_deg = (3 * u.arcmin).to_value(u.deg)

query = f"""
        SELECT host_id, transient_id, transient_classification,
            lc.B_VEGA_MAG, lc.B_VEGA_MAGERR,
            lc.R_AB_MAG, lc.R_AB_MAGERR,
            lc.R_VEGA_MAG, lc.R_VEGA_MAGERR
        FROM safcat.PESSTO_TRAN_CAT_V3 AS master
        JOIN safcat.PESSTO_MPHOT_V3 AS lc ON lc.SOURCE_ID = master.TRANSIENT_ID
        WHERE CONTAINS(
            POINT('', transient_raj2000, transient_decj2000),
            CIRCLE('', {ra_deg}, {dec_deg}, {radius_deg})
        ) = 1
        ORDER BY transient_id
        """

pessto_lightcurve = eso.query_tap(query, tap_endpoint="tap_cat")
pessto_lightcurve[:3]

HOST_ID,TRANSIENT_ID,TRANSIENT_CLASSIFICATION,B_VEGA_MAG,B_VEGA_MAGERR,R_AB_MAG,R_AB_MAGERR,R_VEGA_MAG,R_VEGA_MAGERR
,,,mag,mag,mag,mag,mag,mag
object,object,object,float32,float32,float32,float32,float32,float32
ESO 154- G 010,SN2013fc,SN IIn,--,--,--,--,16.71,0.09
ESO 154- G 010,SN2013fc,SN IIn,--,--,--,--,16.86,0.11
ESO 154- G 010,SN2013fc,SN IIn,--,--,--,--,16.7,0.06


<hr style="border:2px solid #0281c9"> </hr>